<a href="https://colab.research.google.com/github/dineshdabbera/FUTURE_ML_03/blob/main/Resume_screening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Resume Screening & Candidate Ranking System
# Future Interns ML Task 3
# ============================================

import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download stopwords
nltk.download('stopwords')

# ============================================
# Load Dataset
# ============================================
df = pd.read_csv("Resume.csv")

# Select required columns
df = df[['Resume_str', 'Category']]

# ============================================
# Text Cleaning Function
# ============================================
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove stopwords
    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Clean resumes
df['clean_resume'] = df['Resume_str'].apply(clean_text)

# ============================================
# Job Description
# ============================================
job_description = """
Looking for a Data Scientist with experience in Python,
Machine Learning, SQL, Pandas, NumPy, TensorFlow,
Deep Learning and Data Analysis.
"""

job_description = clean_text(job_description)

# ============================================
# TF-IDF Vectorization
# ============================================
documents = [job_description] + list(df['clean_resume'])

vectorizer = TfidfVectorizer(max_features=5000)

tfidf_matrix = vectorizer.fit_transform(documents)

# ============================================
# Similarity Calculation
# ============================================
job_vector = tfidf_matrix[0]

resume_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(
    job_vector,
    resume_vectors
)

# Add similarity scores
df['Similarity Score'] = similarity_scores.flatten()

# ============================================
# Candidate Ranking
# ============================================
ranked_candidates = df.sort_values(
    by='Similarity Score',
    ascending=False
)

print("\n==============================")
print("TOP 10 CANDIDATES")
print("==============================\n")

top_10 = ranked_candidates[
    ['Category', 'Similarity Score']
].head(10)

print(top_10)

# ============================================
# Skill Gap Analysis
# ============================================
required_skills = [
    "python",
    "machine learning",
    "sql",
    "pandas",
    "numpy",
    "tensorflow",
    "deep learning"
]

best_resume = ranked_candidates.iloc[0]['clean_resume']

missing_skills = []

for skill in required_skills:

    if skill not in best_resume:

        missing_skills.append(skill)

# ============================================
# Display Results
# ============================================
print("\n==============================")
print("BEST MATCHED CANDIDATE")
print("==============================")

print("Category :", ranked_candidates.iloc[0]['Category'])

print("Similarity Score :",
      round(ranked_candidates.iloc[0]['Similarity Score']*100,2),
      "%")

print("\n==============================")
print("MISSING SKILLS")
print("==============================")

if len(missing_skills)==0:
    print("No missing skills")
else:
    for skill in missing_skills:
        print("-", skill)

# ============================================
# Save Top 10 Candidates
# ============================================
top_10.to_csv(
    "top_10_candidates.csv",
    index=False
)

print("\nTop 10 candidates saved successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.



TOP 10 CANDIDATES

                   Category  Similarity Score
331  INFORMATION-TECHNOLOGY          0.181193
194                DESIGNER          0.151860
243  INFORMATION-TECHNOLOGY          0.139147
229  INFORMATION-TECHNOLOGY          0.128849
361                 TEACHER          0.127441
337                 TEACHER          0.126300
349                 TEACHER          0.123130
374                 TEACHER          0.121754
315  INFORMATION-TECHNOLOGY          0.121062
280  INFORMATION-TECHNOLOGY          0.119081

BEST MATCHED CANDIDATE
Category : INFORMATION-TECHNOLOGY
Similarity Score : 18.12 %

MISSING SKILLS
- machine learning
- pandas
- numpy
- tensorflow
- deep learning

Top 10 candidates saved successfully!
